# L15 · Repaso: de A1 a A3

**S15 · Lectura: Repaso Caps. 1–15 · 40–55 min**

## La pregunta

¿Cuál es el eslabón más débil de mi sistema causal?

## Qué harás

Revisar identificación, estimación, validación y servicio.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Una auditoría breve de los artefactos y de la bitácora. Incluye una frase sobre qué deja de poder afirmarse cuando la verdad (`tau`, `y0`, `y1`) queda reservada.

> **Idea clave:** La cadena completa es tan fuerte como su supuesto peor tratado. La verdad conocida del DGP sirve para probar el código, no para justificar una decisión en producción.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression
rng = setup()
sc = load_scenario("synthetic/inference/ihdp_medium"); sc.card()
cols = list(sc.covariates)

In [ ]:
# Vista rápida de los datos del escenario
import matplotlib.pyplot as plt
import numpy as np
_fig, _ax = plt.subplots(1, 2, figsize=(9, 3.2))
try:
    _scenario = globals().get("sc") or globals().get("lc")
    if _scenario is None:
        raise NameError("No hay un escenario cargado para el gráfico editorial")
    _df = _scenario.train
    if _scenario.treatment is not None and _scenario.outcome is not None:
        _t = _df[_scenario.treatment].to_numpy(float)
        _y = _df[_scenario.outcome].to_numpy(float)
        if np.unique(_t).size == 2:
            _counts = [int((_t == value).sum()) for value in (0, 1)]
            _ax[0].bar(["control", "tratado"], _counts)
            _ax[0].set_title("Asignación")
            _ax[1].boxplot([_y[_t == 0], _y[_t == 1]], labels=["control", "tratado"])
            _ax[1].set_title("Resultado observado")
        else:
            _ax[0].hist(_t, bins=20)
            _ax[0].set_title(_scenario.treatment)
            _ax[1].scatter(_t, _y, s=8, alpha=.35)
            _ax[1].set_title(f"{_scenario.treatment} frente a {_scenario.outcome}")
    elif _scenario.adj is not None:
        _ax[0].imshow(_scenario.adj, cmap="Greys", interpolation="nearest")
        _ax[0].set_title("Grafo del escenario")
        _ax[1].axis("off")
    else:
        _ax[0].axis("off"); _ax[1].axis("off")
    _fig.suptitle(f"Escenario: {_scenario.name}")
    _fig.tight_layout(); plt.show()
finally:
    plt.close(_fig)

## 1. A1 · Identificación: ¿se puede preguntar esto? (Caps. 1–9)
Tres comprobaciones rápidas, cada una remite a un lab anterior: **balance** (L06), **solapamiento** (L10) y **ausencia de fugas** (L08).

In [ ]:
X, Y, T = sc.xyt("train")
def smd1(x, t):
    a, b = x[t == 1], x[t == 0]; return (a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2 + 1e-12)
bal = pd.Series({c: smd1(X[:, j], T) for j, c in enumerate(cols)}); top = bal.abs().sort_values(ascending=False).head(5)
e_hat = LogisticRegression(max_iter=500).fit(X, T).predict_proba(X)[:, 1]
fuera = float(((e_hat < .05) | (e_hat > .95)).mean())
assert_no_truth(sc.train[cols], "covariables de entrenamiento")           # nada de columnas de verdad entre las covariables
print("mayor |SMD| (balance):", {k: round(v, 2) for k, v in top.items()})
print(f"ê fuera de [0,05; 0,95]: {100 * fuera:.1f} %  ·  columnas de verdad entre las covariables: ninguna ✔")

## 2. A2 · Estimación: ATE con DML (Caps. 10–11, 14)
El artefacto A2 de L11: DML-PLR con cross-fitting. Aquí solo lo recuperamos y lo mostramos.

In [ ]:
art = a2_fit()
print("A2:", fmt_ci(art["theta"], art["se"]), "· n_train =", art["n_train"])
print("hash de los datos:", art["hash_datos_entrenamiento"][:30], "…")

## 3. A3 · Heterogeneidad y servicio (Caps. 12–13, 15)
CATE lineal por R-learner (como en L12) y el archivo de puntajes de prueba que se entregaría (`cate_scores_test`: una fila por unidad de prueba).

In [ ]:
mk = lambda: ExtraTreesRegressor(n_estimators=40, min_samples_leaf=5, max_features=0.8, random_state=0)
n = len(Y); ry = np.zeros(n); rd = np.zeros(n)
for te in kfold(n, 5, SEED):
    tr = np.setdiff1d(np.arange(n), te); ry[te] = Y[te] - mk().fit(X[tr], Y[tr]).predict(X[te]); rd[te] = T[te] - mk().fit(X[tr], T[tr]).predict(X[te])
mu, sd = X.mean(0), X.std(0) + 1e-9; Z = (X - mu) / sd; Am = np.column_stack([rd, rd[:, None] * Z])
coef = np.linalg.solve(Am.T @ Am + np.diag(np.r_[0, np.full(Z.shape[1], 5.0)]), Am.T @ ry)
Xte = sc.test[cols].to_numpy(float); cate = coef[0] + ((Xte - mu) / sd) @ coef[1:]
id_col = "row_id" if "row_id" in sc.test.columns else None
scores = pd.DataFrame({"row_id": sc.test[id_col].to_numpy() if id_col else np.arange(len(Xte)), "cate": cate})
print(scores.shape, "· CATE medio:", scores["cate"].mean().round(3), "· ATE de A2:", round(art["theta"], 3))

## 4. El mapa de supuestos (Completa tú)
Cada capítulo introduce **al menos un supuesto**. Para cada uno decide:
- **tipo:** `observable` (se mide en los datos), `falsificable` (una prueba puede refutarlo, pero no confirmarlo) o `no observable` (solo se declara y se acota);
- **cómo se trató:** `medido`, `contrastado` o `declarado y acotado`.

Regla del curso: un supuesto **no observable** nunca se «mide»; solo se declara y se acota (con una sensibilidad, un E-value, un conocimiento externo).

Completa las celdas `None` del diccionario `mapa` (tuplas `(supuesto, tipo, trato)`). Las tres primeras ya están resueltas como ejemplo.

In [ ]:
# ✏️ Completa tú — mapa de supuestos
# Pista: Pregúntate: ¿puedo comprobarlo con los datos que tengo? Si no, ¿lo declaro y cómo lo acoto?
mapa = {
    1:  ("SUTVA/consistencia: el tratamiento está bien definido y no hay interferencia", "no observable", "declarado y acotado"),
    2:  ("El entorno es reproducible (versiones, semilla, hash)", "observable", "medido"),
    3:  ("El SCM describe el mecanismo (condición de Markov causal)", "no observable", "declarado y acotado"),
    4:  None,   # roles y momento de medición de cada variable
    5:  None,   # ML predictivo vs efecto causal
    6:  None,   # ignorabilidad
    7:  None,   # balance y solapamiento
    8:  None,   # DAG y puerta trasera
    9:  None,   # fugas
    10: None,   # calidad de los nuisance / cross-fitting
    11: None,   # solapamiento estricto
    12: None,   # meta-learners por subgrupo
    13: None,   # estabilidad del bosque
    14: None,   # validación: qué prueba un placebo
    15: None,   # servicio: skew y soporte
}
# Sustituye cada None por una tupla (supuesto, tipo, trato). tipo ∈ {observable, falsificable, no observable};
# trato ∈ {medido, contrastado, declarado y acotado}.

In [ ]:
def validar_mapa(mapa):
    mal = []
    for cap in range(1, 16):
        v = mapa.get(cap)
        if v is None: mal.append(f"Cap. {cap}: sin completar"); continue
        s, tipo, trato = v
        if tipo not in TIPOS or trato not in TRATOS: mal.append(f"Cap. {cap}: tipo/trato fuera del vocabulario")
        elif len(s.strip()) < 10: mal.append(f"Cap. {cap}: describe el supuesto")
        elif tipo == "no observable" and trato != "declarado y acotado": mal.append(f"Cap. {cap}: un supuesto no observable no se «mide» ni se «contrasta»: se declara y se acota")
        elif tipo == "observable" and trato == "declarado y acotado": mal.append(f"Cap. {cap}: si es observable, mídelo")
    return mal
mal = validar_mapa(mapa)
print("✔ mapa completo y coherente" if not mal else "✘ " + " | ".join(mal))
if not mal:
    show(pd.DataFrame(mapa, index=["supuesto", "tipo", "cómo se trató"]).T.rename_axis("Cap.")[["tipo", "cómo se trató", "supuesto"]])
    print(pd.DataFrame(mapa, index=["s", "tipo", "trato"]).T["tipo"].value_counts().to_dict())

**Mira el resultado.** Cuenta cuántos supuestos son «no observable». ¿Qué te dice sobre dónde está el riesgo de un sistema causal, aunque todas las pruebas automáticas pasen?

**Respuesta:**

Los supuestos de identificación (SUTVA, ignorabilidad, DAG, no confusión por subgrupo) son los que **ningún dato puede confirmar**: viven en declaraciones y sensibilidades. Las pruebas automáticas cubren sobre todo los observables y falsificables (balance, solapamiento, fuga, calidad de nuisance, estabilidad). Un sistema con todas las luces verdes sigue dependiendo de que lo no observable esté bien declarado y acotado.



## 5. Auditoría de artefactos: ¿cuál es la brecha más importante?
Tres comprobaciones sobre el paquete de entrega (artefacto A2 + `cate_scores_test`):
1. **orden de características:** `covariables` del artefacto == columnas del escenario, en el mismo orden;
2. **hash:** el hash guardado coincide con el de los datos de entrenamiento;
3. **forma:** una fila por unidad de prueba, sin duplicados ni vacíos.

In [ ]:
def auditar(art_, scores_, sc_):
    c = list(sc_.covariates); t, y = sc_.treatment, sc_.outcome
    f = {}
    f["orden"] = art_["covariables"] == c
    f["hash"]  = art_["hash_datos_entrenamiento"] == sha256_df(sc_.train[c + [t, y]])
    ids = set(sc_.test["row_id"]) if "row_id" in sc_.test.columns else set(range(len(sc_.test)))
    f["forma"] = bool((len(scores_) == len(sc_.test)) and scores_["row_id"].is_unique and scores_["cate"].notna().all() and set(scores_["row_id"]) == ids)
    return f
import copy
def plantar(art_, scores_, tipo):
    a, s = copy.deepcopy(art_), scores_.copy()
    if tipo == "orden": a["covariables"][0], a["covariables"][1] = a["covariables"][1], a["covariables"][0]
    elif tipo == "hash": a["hash_datos_entrenamiento"] = "sha256:" + "0" * 64
    elif tipo == "forma": s = pd.concat([s.iloc[:-3], s.iloc[:3]], ignore_index=True)           # 3 filas duplicadas y 3 filas faltantes
    return a, s
print("paquete sin defecto:", auditar(art, scores, sc))
filas = {}
for tipo in ("orden", "hash", "forma"):
    a2, s2 = plantar(art, scores, tipo); r_ = auditar(a2, s2, sc); filas[tipo] = {k: ("ok" if v else "FALLA") for k, v in r_.items()}
show(pd.DataFrame(filas).T.rename_axis("defecto plantado"))

**Mira el resultado.** Las tres auditorías detectan su defecto. ¿Cuál de los tres es el más peligroso si NO se detecta, y por qué?

**Respuesta:**

El **orden de características**: no produce ningún error, el servicio responde con números plausibles pero equivocados (L12). El hash roto es un problema de trazabilidad (no sabes con qué datos se entrenó), grave pero visible en una auditoría. La forma errónea suele fallar fuerte al unir con los datos reales. Gravedad = daño silencioso × alcance.



### Completa tú: nombra la brecha
El arnés **planta un defecto** en una copia del paquete (según `SEMILLA_DEFECTO`; tú no sabes cuál). Ejecuta la auditoría, identifica qué falla y escribe en `brecha` **una frase** que nombre el defecto (palabras clave: *orden*, *hash* o *forma*) y diga por qué es la más importante.

In [ ]:
SEMILLA_DEFECTO = 15
tipo_plantado = ("orden", "hash", "forma")[SEMILLA_DEFECTO % 3]
art_x, scores_x = plantar(art, scores, tipo_plantado)
res = auditar(art_x, scores_x, sc)
print("resultado de la auditoría:", {k: ("ok" if v else "FALLA") for k, v in res.items()})

In [ ]:
# ✏️ Completa tú — brecha
# Pista: Mira qué clave de `res` es FALSA y usa una palabra clave (orden, hash o forma).
brecha = ""      # TODO: una frase que nombre el defecto (orden / hash / forma) y justifique por qué importa
if not brecha:
    raise NotImplementedError("escribe la brecha en una frase")

In [ ]:
fallos = [k for k, v in res.items() if not v]
assert fallos, "la auditoría no encontró ningún fallo: revisa las comprobaciones"
assert any(k in brecha.lower() for k in fallos), f"tu frase debe nombrar el defecto detectado ({fallos})"
print("✔ brecha identificada:", fallos[0]); print("  ", brecha)

## 6. Supuesto y bitácora

In [ ]:
fila_b = bitacora(
    pregunta="¿Cuál es la brecha más importante del sistema A1→A3 hasta hoy?",
    metodo="Mapa de supuestos Caps. 1–15 + auditoría de orden de características, hash y forma de puntajes",
    estimando="ATE de A2 y CATE de A3 (sin cambios)",
    numero_ic=fmt_ci(art["theta"], art["se"]),
    supuesto=supuesto("observable", "medido", f"la auditoría falla en «{fallos[0]}»; el resto de las comprobaciones pasa"),
    decision="alertar", lab="L15")

---
## Qué queda para el libro
Quedan en el libro: los Caps. 16–18 (sin laboratorio) y todo lo posterior (L17 en adelante). Este repaso no introduce métodos nuevos.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*